In [ ]:
# Analisis exploratorio del test GAIT Arms

# Este cuaderno documenta los pasos para cargar y revisar los datos IMU registrados en BASE-SPINE, LEFT-HAND y RIGHT-HAND durante el test GAIT_ARMS.
# Objetivo del proyecto: caracterizar el comportamiento de un sujeto sano (sin Parkinson)
# para establecer una línea base de estabilidad postural y oscilación de brazos.

In [47]:
# Librerias y ubicacion del archivo
%pip install plotly

import json
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go

from ipywidgets import Dropdown, VBox, HTML

from IPython.display import display



pacient_dir = Path("../../data/posturalTremor/pacient_rojas")

files = sorted([p.name for p in pacient_dir.glob("*.json")])

if not files:

    display(HTML(f"<b>No se encontraron JSON en {pacient_dir}</b>"))

else:

    dd = Dropdown(options=files, description="Archivo:", layout=dict(width="60%"))

    out = HTML()



    def on_change(change):

        if change['name'] == 'value' and change['new']:

            global DATA_PATH

            DATA_PATH = pacient_dir / change['new']

            out.value = f"Seleccionado: <code>{DATA_PATH}</code>"



    dd.observe(on_change, names='value')

    if files:

        # Inicializa con el primero

        DATA_PATH = pacient_dir / files[0]

        out.value = f"Seleccionado: <code>{DATA_PATH}</code>"

    display(VBox([dd, out]))

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 23.0.1 -> 25.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [48]:
# Cargar/Refrescar datos seleccionados
from ipywidgets import Button, HBox, HTML
from IPython.display import display

status = HTML("")

def load_current_file():
    """Carga el JSON apuntado por DATA_PATH y actualiza variables globales (imu_df, patient, test_type)."""
    global imu_df, patient, test_type
    try:
        path = Path(DATA_PATH)
    except NameError:
        status.value = "<span style='color:red'>Define DATA_PATH con el selector primero.</span>"
        return
    if not path.exists():
        status.value = f"<span style='color:red'>No existe el archivo: {path}</span>"
        return

    # Leer JSON
    with path.open(encoding="utf-8") as f:
        raw = json.load(f)

    # Extraer imuData soportando ambas variantes (raíz o dentro de dgiResults)
    imu_entries = []
    if isinstance(raw, dict):
        if raw.get("imuData"):
            imu_entries = raw.get("imuData", [])
        elif isinstance(raw.get("dgiResults"), list):
            for r in raw.get("dgiResults", []):
                if isinstance(r, dict) and r.get("imuData"):
                    imu_entries.extend(r.get("imuData", []))

    patient = raw.get("patient", {}) if isinstance(raw, dict) else {}
    test_type = raw.get("testType") if isinstance(raw, dict) else None

    records = []
    for entry in imu_entries:
        device = entry.get("deviceId") or entry.get("device")
        ts = entry.get("timestamp")
        acc = entry.get("accelerometer", {}) or {}
        gyro = entry.get("gyroscope", {}) or {}
        records.append({
            "device": device,
            "timestamp": ts,
            "acc_x": acc.get("x"),
            "acc_y": acc.get("y"),
            "acc_z": acc.get("z"),
            "gyro_x": gyro.get("x"),
            "gyro_y": gyro.get("y"),
            "gyro_z": gyro.get("z"),
        })

    imu_df = pd.DataFrame.from_records(records)
    if imu_df.empty:
        top_keys = list(raw.keys()) if isinstance(raw, dict) else []
        dgi_count = len(raw.get("dgiResults", [])) if isinstance(raw, dict) and isinstance(raw.get("dgiResults"), list) else 0
        status.value = f"<span style='color:orange'>Cargado {path.name}, pero no se encontraron registros imuData. Claves: {top_keys}, dgiResults: {dgi_count}.</span>"
        return

    imu_df.sort_values(["device", "timestamp"], inplace=True)
    imu_df.reset_index(drop=True, inplace=True)
    imu_df["acc_mag"] = np.sqrt(imu_df["acc_x"] ** 2 + imu_df["acc_y"] ** 2 + imu_df["acc_z"] ** 2)
    imu_df["gyro_mag"] = np.sqrt(imu_df["gyro_x"] ** 2 + imu_df["gyro_y"] ** 2 + imu_df["gyro_z"] ** 2)

    devices = sorted(imu_df["device"].astype(str).unique().tolist())
    # Mostrar resumen de carga + datos del paciente
    status.value = (
        f"Cargado: <b>{path.name}</b> | filas: {len(imu_df):,} | dispositivos: {', '.join(devices)}"
        f"<br><b>Paciente:</b> {patient.get('name','N/D')} (ID: {patient.get('id','N/D')})"
        f"<br><b>Fecha de nacimiento:</b> {patient.get('birthDate','N/D')}"
        f"<br><b>Observaciones:</b> {patient.get('observations') or 'Sin observaciones registradas'}"
    )

btn = Button(description="Cargar/Refrescar datos", button_style="primary")
def _on_click(b):
    load_current_file()
btn.on_click(_on_click)

display(HBox([btn, status]))

In [50]:
# Carga y normalizacion del JSON
with DATA_PATH.open(encoding="utf-8") as f:
    raw = json.load(f)

# Normalizar ubicación de los datos IMU: soportar varias versiones/formatos
# - Formato antiguo/esperado: top-level "imuData"
# - Formato observado en algunos archivos: "dgiResults" -> list -> cada item puede tener "imuData"
# Recolectaremos todos los objetos imuData disponibles en una lista 'imu_entries'.
imu_entries = []
if isinstance(raw, dict):
    # Caso 1: imuData en la raíz
    if raw.get("imuData"):
        imu_entries = raw.get("imuData", [])
    # Caso 2: imuData dentro de dgiResults (p. ej. una lista de resultados)
    elif isinstance(raw.get("dgiResults"), list):
        for r in raw.get("dgiResults", []):
            if isinstance(r, dict) and r.get("imuData"):
                imu_entries.extend(r.get("imuData", []))
    # Otros casos: intentar encontrar cualquier clave que parezca contener "imuData"
    else:
        # No modificar nada aquí; dejamos imu_entries vacío para manejarlo más abajo
        imu_entries = []
else:
    # Si el JSON no es un dict, no sabemos manejarlo
    imu_entries = []

patient = raw.get("patient", {}) if isinstance(raw, dict) else {}
test_type = raw.get("testType") if isinstance(raw, dict) else None

records = []
for entry in imu_entries:
    # some files may use deviceId o device
    device = entry.get("deviceId") or entry.get("device")
    ts = entry.get("timestamp")
    acc = entry.get("accelerometer", {}) or {}
    gyro = entry.get("gyroscope", {}) or {}

    records.append(
        {
            "device": device,
            "timestamp": ts,
            "acc_x": acc.get("x"),
            "acc_y": acc.get("y"),
            "acc_z": acc.get("z"),
            "gyro_x": gyro.get("x"),
            "gyro_y": gyro.get("y"),
            "gyro_z": gyro.get("z"),
        }
    )

imu_df = pd.DataFrame.from_records(records)
if imu_df.empty:
    # Mensaje informativo con pistas para depuración
    top_keys = list(raw.keys()) if isinstance(raw, dict) else []
    dgi_count = len(raw.get("dgiResults", [])) if isinstance(raw, dict) and isinstance(raw.get("dgiResults"), list) else 0
    raise ValueError(
        "No se encontraron registros en imuData. "
        f"Claves en el JSON: {top_keys}. "
        f"dgiResults encontrado: {dgi_count} elementos. "
        "Si los datos IMU están anidados dentro de 'dgiResults', use la versión del código que extrae 'imuData' desde allí o actualice este cuaderno."
    )

# Continuar con el procesamiento usual
imu_df.sort_values(["device", "timestamp"], inplace=True)
imu_df.reset_index(drop=True, inplace=True)
imu_df["acc_mag"] = np.sqrt(imu_df["acc_x"] ** 2 + imu_df["acc_y"] ** 2 + imu_df["acc_z"] ** 2)
imu_df["gyro_mag"] = np.sqrt(imu_df["gyro_x"] ** 2 + imu_df["gyro_y"] ** 2 + imu_df["gyro_z"] ** 2)
imu_df.head()

,device,timestamp,acc_x,acc_y,acc_z,gyro_x,gyro_y,gyro_z,acc_mag,gyro_mag
0,BASE-SPINE,2303,1.072021,0.023315,-0.101440,-2.990723,9.460449,11.47461,1.077062,15.169416
1,BASE-SPINE,2329,1.055664,-0.025391,-0.096802,-7.934570,12.084960,10.25391,1.060397,17.724174
2,BASE-SPINE,2356,1.042603,-0.028442,-0.080811,-9.277344,12.390140,10.68115,1.046117,18.806160
3,BASE-SPINE,2382,1.032959,-0.023682,-0.087891,-10.192870,11.535640,11.16943,1.036962,19.018984
4,BASE-SPINE,2408,1.055908,-0.041016,-0.064575,-13.610840,14.526370,11.96289,1.058676,23.224580


In [51]:
# Resumen del paciente y conteos globales
print(f"Paciente: {patient.get('name', 'N/D')} ({patient.get('id', 'N/D')})")
print(f"Fecha de nacimiento: {patient.get('birthDate', 'N/D')}")
print(f"Observaciones: {patient.get('observations') or 'Sin observaciones registradas'}")
print(f"Tipo de prueba: {test_type}")
print(f"Total de registros IMU: {imu_df.shape[0]:,}")
print(f"Dispositivos disponibles: {sorted(imu_df['device'].unique().tolist())}")

Paciente: Rojas Jojoa Caviedez (12345)
Fecha de nacimiento: 20/10/2025
Observaciones: Rojas Prueba brazos extendidos por 10 segundos
Tipo de prueba: GAIT_ARMS
Total de registros IMU: 3,490
Dispositivos disponibles: ['BASE-SPINE', 'LEFT-HAND', 'RIGHT-HAND']


In [52]:
# Brecha temporal y frecuencia de muestreo aproximada
def summarize_sampling(df: pd.DataFrame) -> pd.DataFrame:
    stats = []
    for device, grp in df.groupby("device"):
        timestamps = grp["timestamp"].astype(float).sort_values()
        deltas = timestamps.diff().dropna()
        stats.append(
            {
                "device": device,
                "samples": len(grp),
                "t_start": timestamps.iloc[0],
                "t_end": timestamps.iloc[-1],
                "duration_ms": timestamps.iloc[-1] - timestamps.iloc[0],
                "mean_dt": deltas.mean() if not deltas.empty else np.nan,
                "median_dt": deltas.median() if not deltas.empty else np.nan,
                "min_dt": deltas.min() if not deltas.empty else np.nan,
                "max_dt": deltas.max() if not deltas.empty else np.nan,
            }
        )
    return pd.DataFrame(stats)

sampling_summary = summarize_sampling(imu_df)
sampling_summary = sampling_summary.assign(
    duration_s=lambda d: d["duration_ms"] / 1000,
    mean_hz=lambda d: 1000 / d["mean_dt"].replace({0: np.nan}),
    median_hz=lambda d: 1000 / d["median_dt"].replace({0: np.nan}),
)
sampling_summary

,device,samples,t_start,t_end,duration_ms,mean_dt,median_dt,min_dt,max_dt,duration_s,mean_hz,median_hz
0,BASE-SPINE,1041,2303.0,30049.0,27746.0,26.678846,26.0,26.0,236.0,27.746,37.482880,38.461538
1,LEFT-HAND,1231,2318.0,35235.0,32917.0,26.761789,26.0,25.0,367.0,32.917,37.366710,38.461538
2,RIGHT-HAND,1218,2344.0,34381.0,32037.0,26.324569,26.0,26.0,79.0,32.037,37.987327,38.461538


In [53]:
# Estadisticas descriptivas por dispositivo
metric_cols = [
    "acc_x",
    "acc_y",
    "acc_z",
    "acc_mag",
    "gyro_x",
    "gyro_y",
    "gyro_z",
    "gyro_mag",
]
desc = imu_df.groupby("device")[metric_cols].agg(["mean", "std", "min", "max"])
desc.columns = [f"{col}_{stat}" for col, stat in desc.columns]
desc.reset_index()

,device,acc_x_mean,acc_x_std,acc_x_min,acc_x_max,acc_y_mean,acc_y_std,acc_y_min,acc_y_max,acc_z_mean,...,gyro_y_min,gyro_y_max,gyro_z_mean,gyro_z_std,gyro_z_min,gyro_z_max,gyro_mag_mean,gyro_mag_std,gyro_mag_min,gyro_mag_max
0,BASE-SPINE,1.026425,0.018028,0.961792,1.147949,-0.089110,0.022882,-0.182495,0.059814,-0.195197,...,-15.6250,19.53125,-0.042918,1.505038,-4.02832,12.3291,2.938439,3.760714,0.086317,26.808887
1,LEFT-HAND,0.125436,0.349131,-0.915161,1.885010,0.296052,0.304964,-0.288818,2.413574,0.815861,...,-130.1880,276.24510,1.531085,28.815149,-126.03760,196.2891,28.221578,53.018125,0.061035,392.416448
2,RIGHT-HAND,0.095973,0.330565,-0.818848,1.706787,-0.112789,0.238618,-1.320679,0.756470,0.929442,...,-186.4014,316.40630,1.374644,25.222203,-162.53660,147.1558,35.466890,66.294144,0.061035,551.311564


In [54]:
# Valores faltantes por dispositivo y variable por si se necesita hacer limpieza
missing = (
    imu_df.drop(columns=["device"])
    .isna()
    .groupby(imu_df["device"])
    .sum()
    .reset_index()
    .rename(columns={"index": "device"})
)
missing

,device,timestamp,acc_x,acc_y,acc_z,gyro_x,gyro_y,gyro_z,acc_mag,gyro_mag
0,BASE-SPINE,0,0,0,0,0,0,0,0,0
1,LEFT-HAND,0,0,0,0,0,0,0,0,0
2,RIGHT-HAND,0,0,0,0,0,0,0,0,0


In [55]:
# Serie temporal de la magnitud de aceleracion
fig_acc = px.line(
    imu_df,
    x="timestamp",
    y="acc_mag",
    color="device",
    title="Magnitud de aceleracion por dispositivo",
    labels={"timestamp": "timestamp (ms)", "acc_mag": "|a| (g)"},
)
fig_acc

In [56]:
# Serie temporal de la magnitud de velocidad angular
fig_gyro = px.line(
    imu_df,
    x="timestamp",
    y="gyro_mag",
    color="device",
    title="Magnitud de velocidad angular por dispositivo",
    labels={"timestamp": "timestamp (ms)", "gyro_mag": "|w| (deg/s)"},
)
fig_gyro

In [57]:
# Componentes individuales del acelerometro
acc_long = imu_df.melt(
    id_vars=["device", "timestamp"],
    value_vars=["acc_x", "acc_y", "acc_z"],
    var_name="axis",
    value_name="acc"
)
fig_acc_axis = px.line(
    acc_long,
    x="timestamp",
    y="acc",
    color="device",
    facet_row="axis",
    title="Componentes del acelerometro",
    labels={"acc": "aceleracion (g)", "timestamp": "timestamp (ms)"},
)
fig_acc_axis.update_layout(showlegend=True)
fig_acc_axis

In [58]:
# Componentes individuales del giroscopio
gyro_long = imu_df.melt(
    id_vars=["device", "timestamp"],
    value_vars=["gyro_x", "gyro_y", "gyro_z"],
    var_name="axis",
    value_name="gyro"
)
fig_gyro_axis = px.line(
    gyro_long,
    x="timestamp",
    y="gyro",
    color="device",
    facet_row="axis",
    title="Componentes del giroscopio",
    labels={"gyro": "velocidad angular (deg/s)", "timestamp": "timestamp (ms)"},
)
fig_gyro_axis.update_layout(showlegend=True)
fig_gyro_axis

## Interpretación de los gráficos actuales
- |a| (magnitud de aceleración): en un sujeto sano, se esperan oscilaciones periódicas con picos regulares (cada paso). La espalda suele tener menor amplitud que las manos.
- |w| (magnitud de velocidad angular): las manos deben mostrar un balanceo armónico con frecuencias cercanas a la cadencia de marcha (≈0.5–1.5 Hz). La espalda presenta variaciones más suaves.
- Ejes individuales: los ejes con mayor energía indican el plano dominante del movimiento (depende de cómo esté orientado el sensor). Simetría entre manos (amplitud y patrón) es un buen signo de marcha equilibrada.

A continuación añadimos métricas cuantitativas (PSD, cadencia, simetría, correlación) para enriquecer la interpretación.

In [59]:
# Utilidades para análisis avanzado (PSD, cadencia, métricas)

import numpy as np

from scipy.signal import welch, correlate



def estimate_fs_ms(timestamps_ms: np.ndarray) -> float:

    ts = np.asarray(timestamps_ms, dtype=float)

    ts = np.sort(ts)

    deltas = np.diff(ts)

    if deltas.size == 0:

        return np.nan

    median_dt = np.median(deltas)

    if median_dt <= 0 or np.isnan(median_dt):

        return np.nan

    return 1000.0 / median_dt  # Hz



def compute_psd(signal: np.ndarray, fs: float, nperseg: int = 512):

    if not np.isfinite(fs) or fs <= 0:

        return np.array([]), np.array([])

    f, pxx = welch(signal, fs=fs, nperseg=min(nperseg, len(signal)))

    return f, pxx



def dominant_freq(signal: np.ndarray, fs: float):

    f, pxx = compute_psd(signal, fs)

    if f.size == 0:

        return np.nan, (np.array([]), np.array([]))

    idx = np.argmax(pxx)

    return f[idx], (f, pxx)



def rms(x: np.ndarray) -> float:

    x = np.asarray(x, dtype=float)

    return float(np.sqrt(np.nanmean(x**2))) if x.size else np.nan



def coef_variation(x: np.ndarray) -> float:

    x = np.asarray(x, dtype=float)

    m = np.nanmean(x)

    s = np.nanstd(x)

    return float(s / m) if m not in (0, np.nan) else np.nan


In [60]:
# PSD y cadencia por dispositivo usando acc_mag y gyro_mag

psd_results = []

cadence_results = []

for device, grp in imu_df.groupby("device"):

    fs = estimate_fs_ms(grp["timestamp"].values)

    # Señales a analizar

    acc_mag = grp["acc_mag"].astype(float).values

    gyro_mag = grp["gyro_mag"].astype(float).values



    # Frecuencia dominante (usaremos acc_mag)

    fdom, (f_acc, pxx_acc) = dominant_freq(acc_mag, fs)

    # Cadencia estimada (pasos/min) ~ 2*picos por ciclo dependiendo, como proxy simple usamos fdom*60

    cadence_spm = fdom * 60 if np.isfinite(fdom) else np.nan



    psd_results.append({"device": device, "fs": fs, "fdom_acc_hz": fdom})

    cadence_results.append({"device": device, "cadence_spm": cadence_spm})



psd_df = pd.DataFrame(psd_results)

cadence_df = pd.DataFrame(cadence_results)

psd_df, cadence_df

(       device         fs  fdom_acc_hz
 0  BASE-SPINE  38.461538     3.906250
 1   LEFT-HAND  38.461538     1.051683
 2  RIGHT-HAND  38.461538     1.051683,
        device  cadence_spm
 0  BASE-SPINE   234.375000
 1   LEFT-HAND    63.100962
 2  RIGHT-HAND    63.100962)

In [61]:
# Graficos: PSD de acc_mag por dispositivo (si se pudo estimar fs)

fig_psd = go.Figure()

for device, grp in imu_df.groupby("device"):

    fs = estimate_fs_ms(grp["timestamp"].values)

    acc_mag = grp["acc_mag"].astype(float).values

    f, pxx = compute_psd(acc_mag, fs)

    if f.size:

        fig_psd.add_trace(go.Scatter(x=f, y=pxx, mode="lines", name=f"{device}"))

fig_psd.update_layout(title="PSD de |a| por dispositivo", xaxis_title="Frecuencia (Hz)", yaxis_title="Potencia")

fig_psd

In [62]:
# Simetría y variabilidad por dispositivo

summary_rows = []

for device, grp in imu_df.groupby("device"):

    row = {

        "device": device,

        "rms_acc": rms(grp["acc_mag"].values),

        "rms_gyro": rms(grp["gyro_mag"].values),

        "cv_acc": coef_variation(grp["acc_mag"].values),

        "cv_gyro": coef_variation(grp["gyro_mag"].values),

    }

    summary_rows.append(row)

metrics_df = pd.DataFrame(summary_rows)

metrics_df

,device,rms_acc,rms_gyro,cv_acc,cv_gyro
0,BASE-SPINE,1.052034,4.771144,0.014695,1.279219
1,LEFT-HAND,1.026755,60.042449,0.158740,1.877875
2,RIGHT-HAND,1.047559,75.161197,0.168092,1.868417


In [63]:
# Correlación cruzada entre manos (si existen ambas manos)

left_aliases = {"LEFT-HAND", "LEFT_HAND", "LEFT", "HAND_LEFT"}

right_aliases = {"RIGHT-HAND", "RIGHT_HAND", "RIGHT", "HAND_RIGHT"}



devices = {d.upper(): d for d in imu_df["device"].astype(str).unique()}

left_key = next((devices[d] for d in devices if d in left_aliases), None)

right_key = next((devices[d] for d in devices if d in right_aliases), None)



if left_key and right_key:

    left = imu_df[imu_df["device"] == left_key].sort_values("timestamp")

    right = imu_df[imu_df["device"] == right_key].sort_values("timestamp")

    # Interpolar a los timestamps comunes para correlación simple

    common_ts = np.intersect1d(left["timestamp"].values, right["timestamp"].values)

    l = left.set_index("timestamp").reindex(common_ts).fillna(method="ffill")["acc_mag"].values

    r = right.set_index("timestamp").reindex(common_ts).fillna(method="ffill")["acc_mag"].values

    l = (l - np.nanmean(l)) / (np.nanstd(l) + 1e-9)

    r = (r - np.nanmean(r)) / (np.nanstd(r) + 1e-9)



    xcorr = correlate(l, r, mode="full")

    lags = np.arange(-len(l)+1, len(l))

    # Lag donde hay máxima correlación -> posible desfase entre manos

    lag_max = lags[np.argmax(xcorr)]

    fig_xcorr = go.Figure(data=go.Scatter(x=lags, y=xcorr, mode="lines"))

    fig_xcorr.update_layout(title=f"Correlación cruzada L/R (lag_max={lag_max})", xaxis_title="lag (muestras)", yaxis_title="correlación")

    fig_xcorr

else:

    print("No se encontraron ambos dispositivos de manos para correlación.")

C:\Users\rojas\AppData\Local\Temp\ipykernel_19768\1086248968.py:27: FutureWarning:

DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.

C:\Users\rojas\AppData\Local\Temp\ipykernel_19768\1086248968.py:29: FutureWarning:

DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.



## Tremor en reposo — protocolo y expectativas

Contexto: sujeto sentado, 5 IMUs (TORSO, LEFT-HAND, RIGHT-HAND, LEFT-FOOT, RIGHT-FOOT). Brazos apoyados en reposabrazos, sin movimiento voluntario.

- En un sujeto sano, el componente espectral en 3–7 Hz (tremor parkinsoniano típico) debería ser bajo o ausente.

- También es frecuente observar banda 8–12 Hz (fisiológica) de muy baja potencia, especialmente en manos.

- Buscaremos: potencia en bandas 3–7 y 8–12 Hz, frecuencia pico en 3–7 Hz (si existe), y coherencia entre manos y entre pies.

In [17]:
# Funciones para tremor en reposo: filtros y bandpower

from scipy.signal import butter, sosfiltfilt, welch, coherence



def bandpass_sos(low, high, fs, order=4):

    if not np.isfinite(fs) or fs <= 0:

        return None

    nyq = fs / 2

    lowc, highc = low / nyq, high / nyq

    if lowc <= 0 or highc >= 1 or lowc >= highc:

        return None

    return butter(order, [lowc, highc], btype="band", output="sos")



def apply_bandpass(x, fs, low, high):

    sos = bandpass_sos(low, high, fs)

    if sos is None:

        return np.array([])

    x = np.asarray(x, dtype=float)

    if x.size < 8:

        return np.array([])

    return sosfiltfilt(sos, x)



def bandpower_welch(x, fs, fmin, fmax):

    if not np.isfinite(fs) or fs <= 0:

        return np.nan

    f, pxx = welch(x, fs=fs, nperseg=min(512, len(x)))

    if f.size == 0:

        return np.nan

    mask = (f >= fmin) & (f <= fmax)

    if not mask.any():

        return 0.0

    # Integración discreta sobre la banda

    df = np.mean(np.diff(f)) if len(f) > 1 else 1.0

    return float(np.trapz(pxx[mask], dx=df))


In [18]:
# Métricas de tremor por dispositivo (bandas 3–7 Hz y 8–12 Hz)

tremor_rows = []

for device, grp in imu_df.groupby("device"):

    fs = estimate_fs_ms(grp["timestamp"].values)

    acc = grp["acc_mag"].astype(float).values

    gyr = grp["gyro_mag"].astype(float).values



    # Bandpower

    bp_3_7_acc = bandpower_welch(acc, fs, 3.0, 7.0)

    bp_8_12_acc = bandpower_welch(acc, fs, 8.0, 12.0)

    bp_3_7_gyr = bandpower_welch(gyr, fs, 3.0, 7.0)

    bp_8_12_gyr = bandpower_welch(gyr, fs, 8.0, 12.0)



    # Pico en 3–7 Hz

    f, pxx = compute_psd(acc, fs)

    if f.size:

        mask = (f >= 3.0) & (f <= 7.0)

        f_peak = float(f[mask][np.argmax(pxx[mask])]) if mask.any() else np.nan

        total_power = float(np.trapz(pxx, x=f))

        band_power = float(np.trapz(pxx[mask], x=f[mask])) if mask.any() else 0.0

        tremor_index = band_power / total_power if total_power > 0 else np.nan

    else:

        f_peak = np.nan

        tremor_index = np.nan



    tremor_rows.append({

        "device": device,

        "fs": fs,

        "bp_3_7_acc": bp_3_7_acc,

        "bp_8_12_acc": bp_8_12_acc,

        "bp_3_7_gyr": bp_3_7_gyr,

        "bp_8_12_gyr": bp_8_12_gyr,

        "f_peak_3_7_acc": f_peak,

        "tremor_index_acc": tremor_index,

    })



tremor_df = pd.DataFrame(tremor_rows)

tremor_df.sort_values("device")

C:\Users\rojas\AppData\Local\Temp\ipykernel_19768\3438397801.py:65: DeprecationWarning:

`trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.

C:\Users\rojas\AppData\Local\Temp\ipykernel_19768\229463577.py:37: DeprecationWarning:

`trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.

C:\Users\rojas\AppData\Local\Temp\ipykernel_19768\229463577.py:39: DeprecationWarning:

`trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.

C:\Users\rojas\AppData\Local\Temp\ipykernel_19768\3438397801.py:65: DeprecationWarning:

`trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functions in `scipy.integrate`.

C:\Users\rojas\AppData\Local\Temp\ipykernel_19768\229463577.py:37: DeprecationWarning:

`trapz` is deprecated. Use `trapezoid` instead, or one of the numerical integration functi

,device,fs,bp_3_7_acc,bp_8_12_acc,bp_3_7_gyr,bp_8_12_gyr,f_peak_3_7_acc,tremor_index_acc
0,BASE-SPINE,38.461538,0.000144,0.000033,1.264625,0.261688,3.530649,0.469636
1,LEFT-ANKLE,25.641026,0.000186,0.000012,1.523753,0.727419,3.104968,0.388168
2,LEFT-HAND,38.461538,0.000095,0.000038,14.957697,3.188835,3.155048,0.388516
3,RIGHT-ANKLE,37.037037,0.000437,0.000563,9.406060,2.510151,6.510417,0.299884
4,RIGHT-HAND,38.461538,0.000105,0.000049,8.393253,3.708633,3.155048,0.378521


In [19]:
# Gráficos: PSD por dispositivo con bandas sombreadas 3–7 y 8–12 Hz

fig_psd_tremor = go.Figure()

for device, grp in imu_df.groupby("device"):

    fs = estimate_fs_ms(grp["timestamp"].values)

    acc = grp["acc_mag"].astype(float).values

    f, pxx = compute_psd(acc, fs)

    if f.size:

        fig_psd_tremor.add_trace(go.Scatter(x=f, y=pxx, mode="lines", name=device))



# Sombras de bandas

fig_psd_tremor.add_vrect(x0=3, x1=7, fillcolor="red", opacity=0.1, line_width=0, annotation_text="3–7 Hz")

fig_psd_tremor.add_vrect(x0=8, x1=12, fillcolor="blue", opacity=0.1, line_width=0, annotation_text="8–12 Hz")

fig_psd_tremor.update_layout(title="PSD |a| con bandas de interés", xaxis_title="Frecuencia (Hz)", yaxis_title="Potencia")

fig_psd_tremor

In [20]:
# Señales filtradas en 3–7 Hz (ventana) para manos y pies

def plot_filtered_window(device_name: str, start_ms: float, duration_ms: float = 5000):

    grp = imu_df[imu_df["device"] == device_name].sort_values("timestamp")

    if grp.empty:

        print(f"No hay datos para {device_name}")

        return None

    fs = estimate_fs_ms(grp["timestamp"].values)

    sig = grp["acc_mag"].astype(float).values

    t = grp["timestamp"].values

    sig_f = apply_bandpass(sig, fs, 3.0, 7.0)

    if sig_f.size == 0:

        print(f"No se pudo filtrar {device_name} (fs inválida o señal corta)")

        return None

    mask = (t >= start_ms) & (t <= start_ms + duration_ms)

    fig = go.Figure()

    fig.add_trace(go.Scatter(x=t[mask], y=sig_f[mask], mode="lines", name=f"{device_name} 3–7 Hz"))

    fig.update_layout(title=f"Señal filtrada 3–7 Hz — {device_name}", xaxis_title="timestamp (ms)", yaxis_title="amplitud (filtrada)")

    return fig



# Ejemplos (ajusta los nombres exactos según tu dataset)

fig_lh = plot_filtered_window("LEFT-HAND", start_ms=float(imu_df["timestamp"].min()))

fig_rh = plot_filtered_window("RIGHT-HAND", start_ms=float(imu_df["timestamp"].min()))

fig_lf = plot_filtered_window("LEFT-FOOT", start_ms=float(imu_df["timestamp"].min()))

fig_rf = plot_filtered_window("RIGHT-FOOT", start_ms=float(imu_df["timestamp"].min()))

fig_lh, fig_rh, fig_lf, fig_rf

No hay datos para LEFT-FOOT
No hay datos para RIGHT-FOOT


(Figure({
     'data': [{'mode': 'lines',
               'name': 'LEFT-HAND 3–7 Hz',
               'type': 'scatter',
               'x': array([], dtype=int64),
               'y': array([], dtype=float64)}],
     'layout': {'template': '...',
                'title': {'text': 'Señal filtrada 3–7 Hz — LEFT-HAND'},
                'xaxis': {'title': {'text': 'timestamp (ms)'}},
                'yaxis': {'title': {'text': 'amplitud (filtrada)'}}}
 }),
 Figure({
     'data': [{'mode': 'lines',
               'name': 'RIGHT-HAND 3–7 Hz',
               'type': 'scatter',
               'x': array([], dtype=int64),
               'y': array([], dtype=float64)}],
     'layout': {'template': '...',
                'title': {'text': 'Señal filtrada 3–7 Hz — RIGHT-HAND'},
                'xaxis': {'title': {'text': 'timestamp (ms)'}},
                'yaxis': {'title': {'text': 'amplitud (filtrada)'}}}
 }),
 None,
 None)

In [21]:
# Coherencia entre manos y entre pies en 3–7 Hz

def coherence_band(signal1, signal2, fs, fmin=3.0, fmax=7.0):

    if not np.isfinite(fs) or fs <= 0:

        return np.nan

    f, Cxy = coherence(signal1, signal2, fs=fs, nperseg=min(512, len(signal1)))

    mask = (f >= fmin) & (f <= fmax)

    if not mask.any():

        return np.nan

    return float(np.nanmean(Cxy[mask]))



def coherence_pair(device_a, device_b):

    ga = imu_df[imu_df["device"] == device_a].sort_values("timestamp")

    gb = imu_df[imu_df["device"] == device_b].sort_values("timestamp")

    common = np.intersect1d(ga["timestamp"].values, gb["timestamp"].values)

    if common.size < 16:

        return np.nan

    fs = estimate_fs_ms(common)

    a = ga.set_index("timestamp").reindex(common).fillna(method="ffill")["acc_mag"].astype(float).values

    b = gb.set_index("timestamp").reindex(common).fillna(method="ffill")["acc_mag"].astype(float).values

    return coherence_band(a, b, fs, 3.0, 7.0)



coh_lh_rh = coherence_pair("LEFT-HAND", "RIGHT-HAND")

coh_lf_rf = coherence_pair("LEFT-FOOT", "RIGHT-FOOT")

print({"coh_LH_RH_3_7": coh_lh_rh, "coh_LF_RF_3_7": coh_lf_rf})

{'coh_LH_RH_3_7': 1.0, 'coh_LF_RF_3_7': nan}


C:\Users\rojas\AppData\Local\Temp\ipykernel_19768\2924311346.py:35: FutureWarning:

DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.

C:\Users\rojas\AppData\Local\Temp\ipykernel_19768\2924311346.py:37: FutureWarning:

DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.



In [22]:
# Diagnóstico: ¿por qué la coherencia de pies dio NaN?

def diagnose_foot_coherence():

    # Detectar nombres reales en el dataset

    devs = {d.upper(): d for d in imu_df["device"].astype(str).unique()}

    foot_left_aliases = {"LEFT-FOOT", "LEFT_FOOT", "FOOT_LEFT", "LEFT-ANKLE", "LEFT_ANKLE"}

    foot_right_aliases = {"RIGHT-FOOT", "RIGHT_FOOT", "FOOT_RIGHT", "RIGHT-ANKLE", "RIGHT_ANKLE"}

    lf = next((devs[k] for k in devs if k in foot_left_aliases), None)

    rf = next((devs[k] for k in devs if k in foot_right_aliases), None)

    print("Dispositivos disponibles:", sorted(devs.values()))

    print("Detectado LEFT-FOOT:", lf, "RIGHT-FOOT:", rf)

    if not (lf and rf):

        print("Algún dispositivo de pie no fue encontrado por nombre. Ajusta los alias arriba según tus etiquetas.")

        return

    ga = imu_df[imu_df["device"] == lf].sort_values("timestamp")

    gb = imu_df[imu_df["device"] == rf].sort_values("timestamp")

    print(f"Samples {lf}:", len(ga), "|", rf, ":", len(gb))

    if ga.empty or gb.empty:

        print("Uno de los pies no tiene datos.")

        return

    common = np.intersect1d(ga["timestamp"].values, gb["timestamp"].values)

    print("Timestamps comunes:", common.size)

    if common.size < 16:

        print("Muy pocas muestras comunes (<16). Coherencia no confiable -> NaN.")

        return

    fs = estimate_fs_ms(common)

    print("fs estimada (Hz):", fs)

    if not np.isfinite(fs) or fs <= 0:

        print("fs inválida (timestamps no permiten estimación). -> NaN")

        return

    a = ga.set_index("timestamp").reindex(common).fillna(method="ffill")["acc_mag"].astype(float).values

    b = gb.set_index("timestamp").reindex(common).fillna(method="ffill")["acc_mag"].astype(float).values

    if np.isnan(a).all() or np.isnan(b).all():

        print("Tras reindex/ffill quedaron NaNs -> coherencia NaN")

        return

    from scipy.signal import coherence as sp_coherence

    f, Cxy = sp_coherence(a, b, fs=fs, nperseg=min(512, len(a)))

    print("Rango de frecuencias calculado: ", (float(f.min()) if f.size else None), "-", (float(f.max()) if f.size else None))

    mask = (f >= 3.0) & (f <= 7.0)

    print("Puntos en banda 3–7 Hz:", int(mask.sum()))

    if not mask.any():

        print("No hay resolución/puntos dentro de 3–7 Hz (fs o nperseg insuficiente). -> NaN")

        return

    band_mean = float(np.nanmean(Cxy[mask]))

    print("Coherencia media 3–7 Hz:", band_mean)



diagnose_foot_coherence()

Dispositivos disponibles: ['BASE-SPINE', 'LEFT-ANKLE', 'LEFT-HAND', 'RIGHT-ANKLE', 'RIGHT-HAND']
Detectado LEFT-FOOT: LEFT-ANKLE RIGHT-FOOT: RIGHT-ANKLE
Samples LEFT-ANKLE: 558 | RIGHT-ANKLE : 658
Timestamps comunes: 23
fs estimada (Hz): 1.5197568389057752
Rango de frecuencias calculado:  0.0 - 0.726840227302762
Puntos en banda 3–7 Hz: 0
No hay resolución/puntos dentro de 3–7 Hz (fs o nperseg insuficiente). -> NaN


C:\Users\rojas\AppData\Local\Temp\ipykernel_19768\2154416468.py:59: FutureWarning:

DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.

C:\Users\rojas\AppData\Local\Temp\ipykernel_19768\2154416468.py:61: FutureWarning:

DataFrame.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.



In [ ]:
# Coherencia 3–7 Hz con resampling a grid uniforme y detección de alias (HAND/ANKLE)
# Alinear por resampling a un grid uniforme usando la fs real de cada dispositivo (no la intersección).
from scipy.signal import coherence as sp_coherence

def resample_to_grid(t_ms: np.ndarray, x: np.ndarray, fs_target: float, t0: float, t1: float):
    if not np.isfinite(fs_target) or fs_target <= 0:
        return None, None
    t_ms = np.asarray(t_ms, dtype=float)
    x = np.asarray(x, dtype=float)
    mask = np.isfinite(t_ms) & np.isfinite(x)
    if mask.sum() < 2:
        return None, None
    step = 1000.0 / fs_target
    # Grid uniforme en la zona de solapamiento
    t_grid = np.arange(t0, t1, step)
    if t_grid.size < 4:
        return None, None
    x_interp = np.interp(t_grid, t_ms[mask], x[mask])
    return t_grid, x_interp

def coherence_pair_resampled(device_a: str, device_b: str, band=(3.0, 7.0)):
    ga = imu_df[imu_df["device"] == device_a].sort_values("timestamp")
    gb = imu_df[imu_df["device"] == device_b].sort_values("timestamp")
    if ga.empty or gb.empty:
        return np.nan

    ts_a = ga["timestamp"].astype(float).values
    ts_b = gb["timestamp"].astype(float).values
    xa = ga["acc_mag"].astype(float).values
    xb = gb["acc_mag"].astype(float).values

    fs_a = estimate_fs_ms(ts_a)
    fs_b = estimate_fs_ms(ts_b)
    if not np.isfinite(fs_a) or not np.isfinite(fs_b) or fs_a <= 0 or fs_b <= 0:
        return np.nan

    fs_tgt = min(fs_a, fs_b)
    # Ventana de solape real
    t0 = max(ts_a.min(), ts_b.min())
    t1 = min(ts_a.max(), ts_b.max())
    if (t1 - t0) < 5000:  # <5 s: poco estable para coherencia
        return np.nan

    t_grid, xa_i = resample_to_grid(ts_a, xa, fs_tgt, t0, t1)
    _,     xb_i = resample_to_grid(ts_b, xb, fs_tgt, t0, t1)
    if t_grid is None or xa_i is None or xb_i is None:
        return np.nan

    # Quitar DC
    xa_i = xa_i - np.nanmean(xa_i)
    xb_i = xb_i - np.nanmean(xb_i)

    nperseg = min(1024, len(t_grid))
    if nperseg < 64:
        return np.nan

    f, Cxy = sp_coherence(xa_i, xb_i, fs=fs_tgt, nperseg=nperseg)
    fmin, fmax = band
    mask = (f >= fmin) & (f <= fmax)
    if not mask.any():
        return np.nan
    return float(np.nanmean(Cxy[mask]))

# Detectar alias reales
devs = {d.upper(): d for d in imu_df["device"].astype(str).unique()}
hand_left_aliases  = {"LEFT-HAND","LEFT_HAND","HAND_LEFT","LEFT"}
hand_right_aliases = {"RIGHT-HAND","RIGHT_HAND","HAND_RIGHT","RIGHT"}
foot_left_aliases  = {"LEFT-FOOT","LEFT_FOOT","FOOT_LEFT","LEFT-ANKLE","LEFT_ANKLE"}
foot_right_aliases = {"RIGHT-FOOT","RIGHT_FOOT","FOOT_RIGHT","RIGHT-ANKLE","RIGHT_ANKLE"}

lh = next((devs[k] for k in devs if k in hand_left_aliases), None)
rh = next((devs[k] for k in devs if k in hand_right_aliases), None)
lf = next((devs[k] for k in devs if k in foot_left_aliases), None)
rf = next((devs[k] for k in devs if k in foot_right_aliases), None)

coh_hands = coherence_pair_resampled(lh, rh) if (lh and rh) else np.nan
coh_feet  = coherence_pair_resampled(lf, rf) if (lf and rf) else np.nan
print({"coh_LH_RH_3_7_resampled": coh_hands, "coh_LF_RF_3_7_resampled": coh_feet})

{'coh_LH_RH_3_7_resampled': 1.0, 'coh_LF_RF_3_7_resampled': 1.0}
